## Импорты и вспомогательные функции
(Эти ячейки надо запустить)

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from collections.abc import Callable
import multiprocessing
import traceback

In [ ]:
# Функции проверок 
def check_dichotomy(dichotomy_method: Callable):
    def f(x):
        return x - np.cos(x)
    # Checking infinite loop
    TIMEOUT = 5  # 5 seconds maximum
    p = multiprocessing.Process(target=dichotomy_method, args=(f, 0, 2))
    p.start()
    p.join(timeout=TIMEOUT)
    if p.is_alive():
        print('Расчет выполняется более 5 секунд. Скорее всего, в программе есть бесконечный цикл')
        p.terminate()
        p.join()
        return
    # Checking if calculations are correct
    ok = True
    try:
        root = dichotomy_method(f, 0, 2)
        if not np.isclose(root, 0.7391, atol=1e-3):
            ok = False
            print(f'Уравнение x - cos(x) = 0:\n\tОжидаемый корень: 0.7391\n\tУ Вас получилось: {root}')
        def f(x):
            return 2**x - 5
        root = dichotomy_method(f, 2, 3)
        if not np.isclose(root, 2.3219, atol=1e-3):
            ok = False
            print(f'Уравнение 2^x - 5 = 0:\n\tОжидаемый корень: 2.3219\n\tУ Вас получилось: {root}')
    except TypeError as e:
        ok = False
        if 'ellipsis' in str(e):
            print('Заполните все троеточия!')
        else:
            traceback.print_exc()
    if ok:
        print('Всё правильно!')

# Решение нелинейных уравнений

Для решения нелинейного уравнения вида 

$$ f(x) = 0 $$

необходимо иметь явно заданную функцию $f(x)$, такую, что для любого аргумента $x$ можно найти значение $f(x)$. В Python такую функцию можно задать через `def`. Например, для функции $f(x)=x^2$:

```
def f(x):
    return x**2
```

## Выделение корней

Для нахождения корней нелинейного уравнения $f(x) = 0$ численно необходимо выделить отрезок $x \in [a, b]$, такой, что выполняются условия:

1. Функция $f(x)$ непрерывна на отрезке $[a, b]$
2. Функция $f(x)$ на концах интервала $[a, b]$ имеет разные знаки, т.е. выполняется условие 
    $$f(a)f(b) < 0$$
3. Производная $f'(x)$ не меняет знак на отрезке $[a, b]$: 
    $$\forall x \in [a, b] \ f'(x)>0, \text{или}$$
    $$\forall x \in [a, b] \ f'(x)<0$$

Проверить выполнимость условий 1 и 3 можно методами математического анализа или визуально (построением графика):

`Пример:` Уравнение $x \ln(x) - 1 = 0$.

В учебном пособии (стр. 101) описан математический способ анализа этой функции на отрезке $[1, e]$. Ниже для наглядности построен график, демонстрирующий функцию на этом отрезке и единственный корень.

In [ ]:
def f(x):
    return x * np.log(x) - 1

_, ax = plt.subplots()
ax.grid(ls='--', lw=1, color='grey')  # Сетка
x = np.linspace(0.9, 5)
y = f(x)
ax.plot(x, y, lw=2)
ax.axvline(1, color='tab:orange')  # Вертикальная линия при x=1
ax.axvline(np.e, color='tab:orange')  # Вертикальная линия при x=e
ax.text(1.1, 6.2, 'x=1', color='tab:orange')
ax.text(2.4, 6.2, 'x=e', color='tab:orange')
ax.annotate('Единственный корень', xy=(1.8, 0.1), xytext=(1.05, 1.5), arrowprops={'arrowstyle': '->'})

## Метод половинного деления (дихотомия)

Пусть уравнение $f(x) = 0$ имеет единственный корень на отрезке $[a, b]$, выполнено условие знакопеременности $f(a)f(b) < 0$.

Алгоритм нахождения корня методом половинного деления (удобная блок-схема приведена на стр. 104 учебного пособия):

1. Найдем точку $c = (a+b)/2$.
2. Проверяем условие знакопеременности для точек $a$ и $c$. Если $f(a)f(c) < 0$, то единственный корень находится между $a$ и $c$. Теперь ищем корень на отрезке $[a, c]$ (заменяем значение переменной $b$ на значение $c$).
3. Если $f(a)f(c) > 0$, то единственный корень находится между $c$ и $b$. Теперь ищем корень на отрезке $[c, b]$ (заменяем значение переменной $a$ на значение $c$).
4. Повторяем пункты 1-3 для нового отрезка $[a, b]$.

### Условие остановки

Для численных методов решения уравнения вводят два условия остановки:

1. Корень должен быть найден с заданной точностью $\varepsilon$, т.е. найденное численно значение корня $x_{num}$ должно отличаться от истинного корня $x$ не более, чем на $\varepsilon$:

$$ |x - x_{num}| < \varepsilon$$

В методе половинного деления в качестве корня берется значение $c = (a+b)/2$, поэтому достаточно потребовать условия 

$$(b - a) < \varepsilon$$

2. Также задают допустимое отклонение $\delta$ значения функции $f(x_{num})$ от нуля:

$$| f(x_{num}) |< \delta$$

В случае метода половинного деления:

$$| f(c) | < \delta$$

### Программирование метода половинного деления

Для удобства введем названия погрешностей:

по $x$: `xtol` = $\varepsilon$

по $y$: `ytol` = $\delta$

(tol) &mdash; от англ. tolerance (погрешность).

Запрограммируйте метод половинного деления согласно алгоритму. На каждом шаге цикла необходимо проверять два условия остановки. Если оба выполнены, цикл должен остановиться.

`Совет` Логическое 'и' в Python для объединения двух условий можно написать так:

```
(условие 1) and (условие 2)
```

In [ ]:
def dichotomy_method(
    f: Callable[[float], float], 
    a: float,
    b: float,
    xtol: float = 1e-4,
    ytol: float = 1e-4
) -> float:
    """Функция для решения нелинейных уравнений методом дихотомии.

    Args:
        f (Callable[[float], float]): Функция левой части уравнения f(x) = 0
        a (float): Левый край отрезка, на котором ведется поиск корня.
        b (float): Правый край отрезка, на котором ведется поиск корня.
        xtol (float, optional): Погрешность по x. Defaults to 1e-4.
        ytol (float, optional): Погрешность по у. Defaults to 1e-4.

    Returns:
        float: Корень уравнения, найденный с заданной точностью.
    """
    # TODO Проверьте, что для a и b выполняется условие знакопеременности. Если не выполнено, функция должна выдать ошибку
    # Обращаться к функции можно так: f(x)
    if ...:
        raise RuntimeError('Условие знакопеременности не выполнено!')
    
    # TODO рассчитайте переменную c (середина отрезка [a, b])
    c = ...
    
    # TODO создайте переменную условия остановки
    stop_condition = (...) and (...)
    
    # Цикл с алгоритмом
    while not stop_condition:
        # TODO Проверьте условие знакопеременности для a и c, замените переменные a или b соответственно.
        ...
        # TODO Рассчитайте новую переменную c
        
        # TODO Обновите условие остановки
        
    return c

# Проверяем
check_dichotomy(dichotomy_method)

# Метод Ньютона
Пусть на интервале $[a, b]$ функция $f(x)$ непрерывна, $f'(x)$ и $f''(x)$ непрерывны и монотонны. 
Пусть $x_n$ - некоторое приближенное значение корня из интервала $[a, b]$.
Точное значение корня можно представить в виде:
$$ \xi = x_n + h_n, $$
где $h_n$ - малая поправка.

Разложим функцию $f(x)$ в точке $x = x_n$ в ряд Тейлора по степеням малой величины $h_n$:
$$ f(x_n + h_n) = f(x_n) + h_n \cdot f'(x_n) $$
Так как $f(x_n + h_n) = f(\xi) = 0$, величина поправки выражается следующим образом:
$$ h_n = -f(x_n)/f'(x_n) $$
Тогда очередное приближение корня можно выразить в виде: 
$$ x_{n+1} = x_n - f(x_n)/f'(x_n) $$

Итерации по данной формуле сходятся к истинному значению корня, если за начальное приближение $x_0$ взять ту границу исходного знакопеременного интервала $[a, b]$, для которой выполняется следующее условие:
$$ f(x_0) \cdot f''(x_0) > 0 $$
(смотрите графическое пояснение на стр. 106 учебного пособия)

## Условие остановки

Оценка погрешности достигнутого приближения искомого корня $ x_n $:

$$ \varepsilon < \frac{M_2}{2m_1} (x_n - x_{n-1})^2, $$
где $M_2$ - максимум абсолютной величины первой производной f''(x) на интервале $[a, b]$
$$M_2 = \underset{a \le x \le b}{\text{max}}|f''(x)|,$$
$m_1$ - минимум абсолютной величины первой производной $f'(x)$ на том же интервале:
$$ m_1= \underset{a \le x \le b}{\text{min}}|f'(x)| $$
Также необходимо проверять условие:
$$ |f(x_n)| < \delta $$

In [ ]:
def newton_method(
    f: Callable[[float], float], 
    a: float,
    b: float,
    xtol: float = 1e-4,
    ytol: float = 1e-4
) -> float:
    """Функция для решения нелинейных уравнений методом Ньютона.

    Args:
        f (Callable[[float], float]): Функция левой части уравнения f(x) = 0
        a (float): Левый край отрезка, на котором ведется поиск корня.
        b (float): Правый край отрезка, на котором ведется поиск корня.
        xtol (float, optional): Погрешность по x. Defaults to 1e-4.
        ytol (float, optional): Погрешность по у. Defaults to 1e-4.

    Returns:
        float: Корень уравнения, найденный с заданной точностью.
    """
    # TODO Проверьте, что для a и b выполняется условие знакопеременности. Если не выполнено, функция должна выдать ошибку
    # Обращаться к функции можно так: f(x)
    if ...:
        raise RuntimeError('Условие знакопеременности не выполнено!')

    # TODO Выберите конец отрезка, с которого необходимо начать расчет

    if  ...:
        x0 = ...
    else:
        x0 = ...
    
    # TODO рассчитайте поправку h_n 
    hn = ...
    
    # TODO создайте переменную условия остановки
    stop_condition = (...) and (...)
    
    # Цикл с алгоритмом
    while not stop_condition:
        # TODO Проверьте условие знакопеременности для a и c, замените переменные a или b соответственно.
        ...
        # TODO Рассчитайте новую переменную c
        
        # TODO Обновите условие остановки
        
    return c

# Проверяем
check_newton(newton_method)

In [ ]:
import sympy as sp
def newton_method(
    f: sp.Expr, 
    a: float,
    b: float,
    xtol: float = 1e-4,
    ytol: float = 1e-4
) -> float:
    """Функция для решения нелинейных уравнений методом Ньютона.

    Args:
        f (Callable[[float], float]): Функция левой части уравнения f(x) = 0
        a (float): Левый край отрезка, на котором ведется поиск корня.
        b (float): Правый край отрезка, на котором ведется поиск корня.
        xtol (float, optional): Погрешность по x. Defaults to 1e-4.
        ytol (float, optional): Погрешность по у. Defaults to 1e-4.

    Returns:
        float: Корень уравнения, найденный с заданной точностью.
    """
    x = sp.symbols('x')
    df = sp.diff(f, x)
    d2f = sp.diff(df, x)

    # TODO Проверьте, что для a и b выполняется условие знакопеременности. Если не выполнено, функция должна выдать ошибку
    # Обращаться к функции можно так: f.subs(x, a)
    if f.evalf(subs = {x: a}) * f.evalf(subs = {x: b}) > 0:
        raise RuntimeError('Условие знакопеременности не выполнено!')

    # TODO Выберите конец отрезка, с которого необходимо начать расчет
    if  f.evalf(subs = {x: a}) * d2f.evalf(subs = {x: a}) > 0:
        x0 = a
    else:
        x0 = b
    
    # TODO рассчитайте поправку h_n 
    hn = - f / df
    m2 = sp.maximum(sp.Abs(d2f), x, sp.Interval(a, b))
    m1 = sp.minimum(sp.Abs(df), x, sp.Interval(a, b))
    xn = x0 - hn
    # TODO создайте переменную условия остановки
    stop_condition = (m2 * (xn.evalf(subs = {x: x0}) > xtol) - x0) ** 2 / (2* m1) and (f.evalf(subs = {x: xn.evalf(subs = {x: x0})}))
    
    # Цикл с алгоритмом
    while not stop_condition:
        # TODO Проверьте условие знакопеременности для a и c, замените переменные a или b соответственно.
        xn = xn - hn
        # TODO Рассчитайте новую переменную c
        
        # TODO Обновите условие остановки
        stop_condition = (m2 * (xn.evalf(subs = {x: x0}) > xtol) - x0) ** 2 / (2* m1) and (f.evalf(subs = {x: xn.evalf(subs = {x: x0})}))
        xn = xn - hn
        
    return xn

In [22]:
import sympy as sp

# Создаём символьную переменную
x = sp.symbols('x')

# Задаем функцию
f = - x**2

# Вычисляем производную
df = sp.diff(f, x)

print(f"Функция: {f}")
print(f"Производная: {df.evalf(subs ={x: 2})}")

Функция: -x**2
Производная: -4.00000000000000


In [26]:
abs(f)

Abs(x**2)

In [23]:
sp.maximum(f, x, sp.Interval(-10, 8))

0